#### 1. local environment

In [2]:
#!/usr/bin/env python3
"""
visualize_imputation.py
WaveStitch+ Imputation Quality Visualization
Evaluates imputed output against ground truth on masked (held-out) regions.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Optional

import matplotlib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

# ─────────────────────────────────────────────
# Global academic-style plotting configuration
# ─────────────────────────────────────────────
matplotlib.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.color": "#D9D9D9",
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "grid.alpha": 0.6,
    "savefig.facecolor": "white",
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.unicode_minus": False,
})

COLORS = {
    "observed": "#1f77b4",        # muted blue
    "true_missing": "#d62728",    # muted red
    "masked": "#7f7f7f",          # grey
    "wavestitchplus": "#ff7f0e",            # orange
    "gt": "#4d4d4d",              # dark grey
}

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────
# DATASET = "python"
# DATASET = "golang"
# DATASET = "amf"
DATASET = "rabbitmq"

BASE_DIR = Path("./work/EUR")
PREPARED_DIR = BASE_DIR / f"prepared_{DATASET}"
CURATED_DIR = BASE_DIR / f"generated_{DATASET}"
OUTPUT_DIR = CURATED_DIR / "comparison_plots"

EVALUATED_FILE = "wavestitchPlus_full_imputed.csv"

# Optional display tweaks
SHOW_PLOTS = False
SAVE_PDF = True
SAVE_PNG = True
PNG_DPI = 300


# ─────────────────────────────────────────────
# Data loading
# ─────────────────────────────────────────────

def load_data() -> tuple[pd.DataFrame, Optional[pd.DataFrame], pd.DataFrame, dict]:
    """Load metadata, test input, ground truth, and WaveStitch+ predictions."""
    meta_path = PREPARED_DIR / "meta.json"
    raw_path = PREPARED_DIR / "test_input.csv"
    gt_path = PREPARED_DIR / "test_gt.csv"
    pred_path = CURATED_DIR / EVALUATED_FILE

    if not meta_path.exists():
        raise FileNotFoundError(f"Missing file: {meta_path}")
    if not raw_path.exists():
        raise FileNotFoundError(f"Missing file: {raw_path}")
    if not pred_path.exists():
        raise FileNotFoundError(f"Missing file: {pred_path}")

    with meta_path.open("r", encoding="utf-8") as f:
        meta = json.load(f)

    raw = pd.read_csv(raw_path)
    gt = pd.read_csv(gt_path) if gt_path.exists() else None
    pred = pd.read_csv(pred_path)

    print(f"\n{'='*60}")
    print(f"Dataset : {DATASET}")
    print(f"Target  : {meta.get('target_cols', [])}")
    print(f"{'='*60}")
    print(f"test_input  : {raw.shape}")
    print(f"WaveStitch+ : {pred.shape}")
    if gt is not None:
        print(f"ground truth: {gt.shape}")
    print()

    return raw, gt, pred, meta


# ─────────────────────────────────────────────
# Time parsing
# ─────────────────────────────────────────────

def parse_time(df: pd.DataFrame, time_col: str) -> pd.DatetimeIndex:
    """Parse time column robustly; fallback to synthetic index if needed."""
    if time_col not in df.columns:
        return pd.to_datetime(range(len(df)), unit="s")

    s = df[time_col]

    if pd.api.types.is_numeric_dtype(s):
        s_non_na = s.dropna()
        if len(s_non_na) == 0:
            return pd.to_datetime(range(len(df)), unit="s")

        max_val = s_non_na.max()
        unit = "ms" if max_val > 1e12 else "s"
        return pd.to_datetime(s, unit=unit, errors="coerce")

    parsed = pd.to_datetime(s, errors="coerce")
    if parsed.isna().all():
        return pd.to_datetime(range(len(df)), unit="s")
    return parsed


def setup_time_axis(ax: plt.Axes, time_index: pd.DatetimeIndex) -> None:
    """Choose a readable formatter based on overall time span."""
    ts = pd.Series(time_index).dropna()
    if len(ts) == 0:
        return

    tmin = ts.min()
    tmax = ts.max()
    duration = tmax - tmin

    if duration.days > 30:
        fmt = "%Y-%m-%d"
    elif duration.days > 1:
        fmt = "%m-%d %H:%M"
    elif duration.total_seconds() > 3600:
        fmt = "%H:%M"
    else:
        fmt = "%H:%M:%S"

    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")


# ─────────────────────────────────────────────
# Mask classification
# ─────────────────────────────────────────────

def classify_mask(
    raw: pd.DataFrame,
    gt: Optional[pd.DataFrame],
    feature: str,
) -> np.ndarray:
    """
    Classify each timestep as one of:
      'observed'     — value present in test input
      'true_missing' — real gap (is_gap=True) or GT also missing
      'masked'       — held-out for evaluation (GT available, input NaN)
    """
    n = len(raw)
    mask_type = np.full(n, "observed", dtype=object)

    is_gap = (
        raw["is_gap"].fillna(0).to_numpy().astype(bool)
        if "is_gap" in raw.columns else np.zeros(n, dtype=bool)
    )
    input_nan = (
        raw[feature].isna().to_numpy()
        if feature in raw.columns else np.ones(n, dtype=bool)
    )
    gt_present = (
        ~gt[feature].isna().to_numpy()
        if gt is not None and feature in gt.columns
        else np.zeros(n, dtype=bool)
    )

    for i in range(n):
        if not input_nan[i]:
            mask_type[i] = "observed"
        elif is_gap[i] or not gt_present[i]:
            mask_type[i] = "true_missing"
        else:
            mask_type[i] = "masked"

    return mask_type


# ─────────────────────────────────────────────
# Metrics
# ─────────────────────────────────────────────

def compute_metrics(
    raw: pd.DataFrame,
    gt: Optional[pd.DataFrame],
    pred: pd.DataFrame,
    feature: str,
) -> dict:
    """Compute MSE, RMSE, and MAE on masked (held-out) positions only."""
    if gt is None or feature not in gt.columns or feature not in pred.columns:
        return {"mse": None, "mae": None, "rmse": None, "n": 0}

    mask_type = classify_mask(raw, gt, feature)
    idx = mask_type == "masked"
    n = int(idx.sum())

    if n == 0:
        return {"mse": None, "mae": None, "rmse": None, "n": 0}

    gt_vals = gt[feature].to_numpy()[idx]
    pred_vals = pred[feature].to_numpy()[idx]
    valid = ~(np.isnan(gt_vals) | np.isnan(pred_vals))

    if valid.sum() == 0:
        return {"mse": None, "mae": None, "rmse": None, "n": 0}

    err = gt_vals[valid] - pred_vals[valid]
    mse = float(np.mean(err ** 2))
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(mse))

    return {"mse": mse, "mae": mae, "rmse": rmse, "n": n}


# ─────────────────────────────────────────────
# Summary printing
# ─────────────────────────────────────────────

def print_mask_summary(
    raw: pd.DataFrame,
    gt: Optional[pd.DataFrame],
    target_cols: list[str],
) -> None:
    print(f"\n{'='*72}")
    print(f"{'Feature':<22} {'Observed':>10} {'True Missing':>14} {'Masked':>10}")
    print(f"{'='*72}")
    for feat in target_cols:
        if feat not in raw.columns:
            continue
        mt = classify_mask(raw, gt, feat)
        print(f"{feat:<22} {(mt == 'observed').sum():>10} "
              f"{(mt == 'true_missing').sum():>14} {(mt == 'masked').sum():>10}")
    print(f"{'='*72}\n")


def print_metrics_summary(
    raw: pd.DataFrame,
    gt: Optional[pd.DataFrame],
    pred: pd.DataFrame,
    target_cols: list[str],
) -> list[dict]:
    print(f"\n{'='*72}")
    print("Imputation Metrics on Masked Regions (WaveStitch+)")
    print(f"{'='*72}")
    print(f"{'Feature':<22} {'MSE':>12} {'RMSE':>12} {'MAE':>12} {'N':>8}")
    print(f"{'-'*72}")

    rows = []
    for feat in target_cols:
        m = compute_metrics(raw, gt, pred, feat)
        mse_s = f"{m['mse']:.6f}" if m["mse"] is not None else "N/A"
        rmse_s = f"{m['rmse']:.6f}" if m["rmse"] is not None else "N/A"
        mae_s = f"{m['mae']:.6f}" if m["mae"] is not None else "N/A"

        print(f"{feat:<22} {mse_s:>12} {rmse_s:>12} {mae_s:>12} {m['n']:>8}")

        rows.append({
            "Feature": feat,
            "MSE": m["mse"],
            "RMSE": m["rmse"],
            "MAE": m["mae"],
            "N_Masked": m["n"],
        })

    print(f"{'='*72}\n")
    return rows


# ─────────────────────────────────────────────
# Background helper
# ─────────────────────────────────────────────

def _add_background_spans(
    ax: plt.Axes,
    mask_type: np.ndarray,
    time_index: pd.DatetimeIndex,
    alpha: float = 0.08,
) -> None:
    """Shade true_missing and masked regions with light academic-style spans."""
    for i in range(len(mask_type) - 1):
        t_start = time_index[i]
        t_end = time_index[i + 1]
        if pd.isna(t_start) or pd.isna(t_end):
            continue

        if mask_type[i] == "true_missing":
            ax.axvspan(t_start, t_end, alpha=alpha,
                       color=COLORS["true_missing"], linewidth=0)
        elif mask_type[i] == "masked":
            ax.axvspan(t_start, t_end, alpha=alpha,
                       color=COLORS["masked"], linewidth=0)


# ─────────────────────────────────────────────
# Plotting
# ─────────────────────────────────────────────

def plot_feature(
    feature: str,
    raw: pd.DataFrame,
    gt: Optional[pd.DataFrame],
    pred: pd.DataFrame,
    time_index: pd.DatetimeIndex,
    metrics: dict,
) -> None:
    mask_type = classify_mask(raw, gt, feature)
    observed_mask = mask_type == "observed"
    masked_mask = mask_type == "masked"
    true_miss_mask = mask_type == "true_missing"

    raw_values = (
        raw[feature].to_numpy()
        if feature in raw.columns else np.full(len(raw), np.nan)
    )

    fig, (ax_bar, ax_main) = plt.subplots(
        2, 1,
        figsize=(16, 6.5),
        sharex=True,
        gridspec_kw={"height_ratios": [0.18, 1]},
    )

    # ── Top mask strip ───────────────────────
    for i in range(len(mask_type) - 1):
        if pd.isna(time_index[i]) or pd.isna(time_index[i + 1]):
            continue

        if mask_type[i] == "true_missing":
            color = COLORS["true_missing"]
        elif mask_type[i] == "masked":
            color = COLORS["masked"]
        else:
            color = "white"

        ax_bar.axvspan(
            time_index[i], time_index[i + 1],
            alpha=0.5, color=color, linewidth=0
        )

    ax_bar.set_ylim(0, 1)
    ax_bar.set_yticks([])
    ax_bar.set_facecolor("white")
    ax_bar.spines[["top", "right", "left", "bottom"]].set_visible(False)
    ax_bar.grid(False)
    ax_bar.legend(
        handles=[
            Patch(facecolor=COLORS["true_missing"], alpha=0.5, label="True missing"),
            Patch(facecolor=COLORS["masked"], alpha=0.5, label="Masked (evaluation)"),
        ],
        loc="upper right",
        ncol=2,
        frameon=True,
    )
    ax_bar.set_title(
        f"{feature}    |    True missing: {true_miss_mask.sum()}    Masked: {masked_mask.sum()}",
        loc="left",
        pad=4,
    )

    # ── Main panel ───────────────────────────
    _add_background_spans(ax_main, mask_type, time_index, alpha=0.08)

    if feature in pred.columns:
        ax_main.plot(
            time_index,
            pred[feature].to_numpy(),
            color=COLORS["wavestitchplus"],
            linewidth=1.5,
            alpha=0.9,
            label="WaveStitch+",
            zorder=2,
        )

    ax_main.scatter(
        time_index[observed_mask],
        raw_values[observed_mask],
        s=8,
        color=COLORS["observed"],
        alpha=0.75,
        label="Observed",
        zorder=3,
    )

    if gt is not None and feature in gt.columns:
        gt_vals = gt[feature].to_numpy()
        ax_main.scatter(
            time_index[masked_mask],
            gt_vals[masked_mask],
            s=24,
            color=COLORS["gt"],
            alpha=0.95,
            marker="x",
            linewidths=1.2,
            label="Ground truth (masked)",
            zorder=5,
        )

    mse_s = f"{metrics['mse']:.5f}" if metrics["mse"] is not None else "N/A"
    rmse_s = f"{metrics['rmse']:.5f}" if metrics["rmse"] is not None else "N/A"
    mae_s = f"{metrics['mae']:.5f}" if metrics["mae"] is not None else "N/A"

    ax_main.text(
        0.01, 0.98,
        f"MSE={mse_s}   RMSE={rmse_s}   MAE={mae_s}   N={metrics['n']}",
        transform=ax_main.transAxes,
        va="top",
        ha="left",
        fontsize=8.5,
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="#BBBBBB",
            alpha=0.9,
        ),
    )

    ax_main.set_ylabel(feature)
    ax_main.set_xlabel("Time")
    ax_main.legend(
        loc="upper right",
        frameon=True,
        handlelength=2.0,
    )
    setup_time_axis(ax_main, time_index)

    plt.tight_layout()

    png_path = OUTPUT_DIR / f"{feature}_wavestitchplus.png"
    pdf_path = OUTPUT_DIR / f"{feature}_wavestitchplus.pdf"

    if SAVE_PNG:
        plt.savefig(png_path, dpi=PNG_DPI)
        print(f"  [SAVED] {png_path}")
    if SAVE_PDF:
        plt.savefig(pdf_path)
        print(f"  [SAVED] {pdf_path}")

    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    raw, gt, pred, meta = load_data()

    time_col = meta.get("time_col", "time")
    target_cols = meta.get("target_cols", [])

    time_index = parse_time(raw, time_col)
    ts = pd.Series(time_index).dropna()
    if len(ts) > 0:
        print(f"Time range: {ts.min()}  →  {ts.max()}\n")
    else:
        print("Time range: invalid / unavailable\n")

    print_mask_summary(raw, gt, target_cols)
    metric_rows = print_metrics_summary(raw, gt, pred, target_cols)

    print("[PLOTTING] Generating per-feature comparison plots...")
    for feat in target_cols:
        if feat not in raw.columns:
            print(f"  [SKIP] '{feat}' not in test_input.csv")
            continue

        m = compute_metrics(raw, gt, pred, feat)
        plot_feature(feat, raw, gt, pred, time_index, m)

    if metric_rows:
        results_df = pd.DataFrame(metric_rows)
        csv_path = OUTPUT_DIR / "metrics_summary.csv"
        results_df.to_csv(csv_path, index=False)
        print(f"\n[SAVED] Metrics table: {csv_path}")
        print(results_df.to_string(index=False))

    print(f"\n[DONE] All plots saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


Dataset : rabbitmq
Target  : ['cpu_limit', 'cpu_usage', 'lat50_ms', 'lat75_ms', 'lat95_ms', 'lat99_ms', 'min_ms', 'n', 'ram_limit_mb', 'ram_usage_mb']
test_input  : (2612, 17)
WaveStitch+ : (2612, 17)
ground truth: (2612, 17)

Time range: 2022-03-22 23:24:59  →  2022-03-25 01:27:38


Feature                  Observed   True Missing     Masked
cpu_limit                     936           1519        157
cpu_usage                     936           1519        157
lat50_ms                      936           1519        157
lat75_ms                      936           1519        157
lat95_ms                      936           1519        157
lat99_ms                      936           1519        157
min_ms                        936           1519        157
n                             936           1519        157
ram_limit_mb                  936           1519        157
ram_usage_mb                  936           1519        157


Imputation Metrics on Masked Regions (WaveStitch+)
F

#### 2. remote 